NOTE: This is an altered copy of a notebook provided by the organizers of the Current Topics in Digital Philology course.

### **Current Topics in Digital Philology 5LN720/5LN721**

# **Lab 5: Assignment on Digital Methods in Literary Analysis**



## **1. Get started**

First, we import the Gutenberg corpus and some packages and tools that might be useful.

In [1]:
import nltk

from nltk.corpus import gutenberg  # The Gutenberg corpus
from nltk.corpus import (
    stopwords,
)  # Stop words = a set of high frequent words in a language (e.g. “the”, “is”, “and”) that you might want to filter out
from nltk import (
    word_tokenize,
    sent_tokenize,
)  # NLTK package for tokenizing words or sentences
from nltk import WordNetLemmatizer  # NLTK package for lemmatizing words

import string
from collections import OrderedDict
import matplotlib.pyplot as plt
from collections import Counter
import re

nltk.download("gutenberg")
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("averaged_perceptron_tagger")
nltk.download("universal_tagset")
nltk.download("wordnet")
nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger_eng")

print("Done!")

Done!


[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\Daan\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Daan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Daan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Daan\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package universal_tagset to
[nltk_data]     C:\Users\Daan\AppData\Roaming\nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Daan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-

## **3. Data Selection**

In [2]:
books = [
    "carroll-alice.txt",
    "austen-sense.txt",
]  # <------------------------------------------------  1. CHANGE TO THE BOOK(S) OF YOUR COICE!

data = []
data_sents = []
for book in books:
    data.append(gutenberg.words(book))  # data as tokens
    data_sents.append(gutenberg.sents(book))  # data as sentences, used for pos tagging

print("Example of how the data as tokens looks like:")
print((" ".join(data_sents[0][0]).replace("[ ", "").replace(" ]", "")))

train_set = [
    "austen-emma.txt",
    "chesterton-ball.txt",
    "shakespeare-caesar.txt",
]

val_set = [
    "austen-persuasion.txt",
    "chesterton-brown.txt",
    "shakespeare-hamlet.txt",
]

test_set = [
    "austen-sense.txt",
    "chesterton-thursday.txt",
    "shakespeare-macbeth.txt",
]

Example of how the data as tokens looks like:
Alice ' s Adventures in Wonderland by Lewis Carroll 1865


# **Custom Code Intermission**

In [3]:
import os
from copy import deepcopy
import json

import pandas as pd
import matplotlib.pyplot as plt

from featurizer import Featurizer
from classifier import Classifier


RERUN_FEATURES = False


def main():
    featurizer = Featurizer()

    full_train_set = get_features(
        featurizer,
        RERUN_FEATURES,
        feature_filename="data/train_features.csv",
        filename=train_set,
    )
    full_dev_set = get_features(
        featurizer,
        RERUN_FEATURES,
        feature_filename="data/dev_features.csv",
        filename=val_set,
    )
    full_test_set = get_features(
        featurizer,
        RERUN_FEATURES,
        feature_filename="data/test_features.csv",
        filename=test_set,
    )

    print(full_train_set["author"].value_counts())

    # exit()
    print("Running classifier...")
    classifier = Classifier()

    colnames = [col for col in full_train_set.columns if col.startswith("f_")]

    train_x = full_train_set[colnames].to_numpy()
    train_y = full_train_set["author"].to_numpy()
    classifier.fit(train_x, train_y)

    print("Evaluating classifier...")
    train_report = classifier.evaluate(train_x, train_y)

    dev_x = full_dev_set[colnames].to_numpy()
    dev_y = full_dev_set["author"].to_numpy()
    dev_report = classifier.evaluate(dev_x, dev_y)

    test_x = full_test_set[colnames].to_numpy()
    test_y = full_test_set["author"].to_numpy()
    test_report = classifier.evaluate(test_x, test_y)

    report = {
        "train_report": train_report,
        "dev_report": dev_report,
        "test_report": test_report,
    }
    with open("classifier_results.json", "w") as outfile:
        json.dump(report, outfile, indent=4)

    print("Doing ablation study")
    perform_ablation_study(colnames, full_train_set, full_dev_set, full_test_set)
    # perform_ablation_study(
    #     colnames, full_train_set, full_dev_set, full_test_set, reverse=True
    # )
    # perform_ablation_study(
    #     colnames, full_train_set, full_dev_set, full_test_set, groups=True
    # )
    # perform_ablation_study(
    #     colnames, full_train_set, full_dev_set, full_test_set, groups=True, reverse=True
    # )


def get_features(
    featurizer: Featurizer, rerun_features: bool, feature_filename: str, filename: str
) -> pd.DataFrame:
    """Get the features for texts and return a dataframe with the text and the features

    Args:
        featurizer (Featurizer): Object that performs featurization
        rerun_features (bool): If the features should be re-calculated
        feature_filename (str): Filename for the calculated features
        filename (str): Filename containing the texts and authors

    Returns:
        pd.Dataframe: All texts, their features and their author
    """
    # data_set = pd.read_csv(filename, index_col=0)
    # data_set.index.names = ["index"]

    if rerun_features or not os.path.exists(feature_filename):
        print("Calculating features...")
        features = featurizer.featurize(filename)
        features.to_csv(feature_filename)
        summary = features.describe(include="all")
        summary.to_csv(f"{feature_filename.split('.')[0].split('_')[0]}_summary.csv")
    else:
        print("Loading features...")
        features = pd.read_csv(feature_filename)

    return features


def perform_ablation_study(
    colnames: list,
    full_train_set: pd.DataFrame,
    full_dev_set: pd.DataFrame,
    full_test_set: pd.DataFrame,
    groups: bool = False,
    reverse: bool = False,
) -> None:
    """Perform an ablation study

    Args:
        colnames (list): Names of the feature columns in the datasets
        full_train_set (pd.DataFrame): Train set
        full_dev_set (pd.DataFrame): Dev set
        full_test_set (pd.DataFrame): Test set
        groups (bool, optional): If the features should be grouped. Defaults to False.
        reverse (bool, optional): When true, the classifiers are trained for each individual feature(group),
            insead of leaving out that feature(group). Defaults to False.
    """
    train_results = {}
    dev_results = {}
    test_results = {}
    train_y = full_train_set["author"].to_numpy()
    dev_y = full_dev_set["author"].to_numpy()
    test_y = full_test_set["author"].to_numpy()
    features = ["d", "c", "p", "g", "i", "o"] if groups else colnames
    for feature in features:
        if groups:
            if reverse:
                ablation_colnames = [
                    colname for colname in colnames if colname[2] == feature
                ]
            else:
                ablation_colnames = [
                    colname for colname in colnames if colname[2] != feature
                ]
        else:
            if reverse:
                ablation_colnames = [feature]
            else:
                ablation_colnames = deepcopy(colnames)
                ablation_colnames.remove(feature)

        train_x = full_train_set[ablation_colnames].to_numpy()
        dev_x = full_dev_set[ablation_colnames].to_numpy()
        test_x = full_test_set[ablation_colnames].to_numpy()

        classifier = Classifier()
        classifier.fit(train_x=train_x, train_y=train_y)

        train_results[feature] = classifier.get_f1_score(eval_x=train_x, eval_y=train_y)
        dev_results[feature] = classifier.get_f1_score(eval_x=dev_x, eval_y=dev_y)
        test_results[feature] = classifier.get_f1_score(eval_x=test_x, eval_y=test_y)

    settings_str = f"{'_groups' if groups else ''}{'_reverse' if reverse else ''}"
    results = {
        "train_results": train_results,
        "dev_results": dev_results,
        "test_results": test_results,
    }
    with open(f"results{settings_str}.json", "w") as outfile:
        json.dump(results, outfile, indent=4)

    plot_ablation_results(train_results, name="Train", settings_str=settings_str)
    plot_ablation_results(dev_results, name="Dev", settings_str=settings_str)
    plot_ablation_results(test_results, name="Test", settings_str=settings_str)


def plot_ablation_results(
    ablation_results: dict, name: str = "", settings_str: str = ""
) -> None:
    """Plot the results of an ablation study

    Args:
        ablation_results (dict): Results of the ablation study. Keys are feature(group) names. Values are F1 scores
        name (str, optional): Name of the dataset the classifiers were evaluated on. Defaults to "".
        settings_str (str, optional): Settings of the ablation study. Defaults to "".
    """

    if "groups" in settings_str:
        x_ticks = [
            "Default Counts",
            "Complexity",
            "POS tags",
            "Grammar/spelling",
            "Punctuation",
            "Other",
        ]
        rotation = 45
        bottom = 0.3
        figsize = (7, 5)
    else:
        x_ticks = [i[4:] for i in ablation_results.keys()]
        rotation = 90
        bottom = 0.45
        figsize = (10, 5)

    plt.figure(figsize=figsize)
    plt.bar(x_ticks, ablation_results.values())
    plt.xticks(rotation=rotation)
    plt.xlabel("Missing feature")
    plt.ylabel("F1 score")
    plt.title(
        f"{name} - Ablation study for authorship attribution ({' '.join(settings_str.split('_')).strip()})"
    )
    plt.subplots_adjust(top=0.9, bottom=bottom)
    plt.savefig(f"ablation_plot_{name.lower()}{settings_str}.png")
    plt.close()


main()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Daan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Daan\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package universal_tagset to
[nltk_data]     C:\Users\Daan\AppData\Roaming\nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!


Calculating features...


100%|██████████| 3/3 [00:13<00:00,  4.63s/it]


Calculating features...


100%|██████████| 3/3 [00:14<00:00,  4.93s/it]


Calculating features...


100%|██████████| 3/3 [00:13<00:00,  4.51s/it]


author
austen         3000
chesterton     3000
shakespeare    2163
Name: count, dtype: int64
Running classifier...
Evaluating classifier...
Accuracy: 0.7304912409653314
F1 score: 0.7298046542128894
Classification report:               precision    recall  f1-score   support

      austen       0.69      0.65      0.67      3000
  chesterton       0.68      0.71      0.70      3000
 shakespeare       0.86      0.87      0.86      2163

    accuracy                           0.73      8163
   macro avg       0.74      0.74      0.74      8163
weighted avg       0.73      0.73      0.73      8163

Accuracy: 0.6766666666666666
F1 score: 0.6779683378383651
Classification report:               precision    recall  f1-score   support

      austen       0.58      0.56      0.57      3000
  chesterton       0.61      0.66      0.63      3000
 shakespeare       0.85      0.81      0.83      3000

    accuracy                           0.68      9000
   macro avg       0.68      0.68      0.68  